# Examen ICCD753 Recuperación de Información

## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

## 2. Corpus asignado

Cada estudiante recibirá un conjunto de datos. Todos contienen documentos, consultas y qrels. Se utilizarán los identificadores originales de los documentos para que los resultados puedan evaluarse.

| Corpus | Documentos y consultas | Qrels |
| --- | --- | --- |
| NFCorpus | [BeIR/nfcorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | [BeIR/nfcorpus-qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) |
| SciFact | [BeIR/scifact](https://huggingface.co/datasets/BeIR/scifact) | [BeIR/scifact-qrels](https://huggingface.co/datasets/BeIR/scifact-qrels) |

## 3. Requisitos del sistema

### 3.1. Preparación e indexación

El sistema deberá:

1. Cargar e indexar los documentos del corpus asignado.
2. Aplicar y documentar las decisiones de preprocesamiento: tokenización, normalización, eliminación de stopwords y stemming o lematización.
3. Construir un vocabulario y un índice invertido que permita recuperar documentos.
4. Conservar la relación entre cada documento indexado y su identificador original.

La unidad de evaluación será el documento original, porque los qrels relacionan consultas con identificadores de documentos. No se debe dividir el texto en fragmentos.

### 3.2. Modelos de recuperación

Para cualquier consulta de texto, el sistema deberá producir dos rankings independientes:

- TF-IDF con similitud coseno.
- BM25.

Cada ranking deberá ordenar los documentos por puntuación de relevancia y permitir solicitar los primeros `k` resultados.

### 3.3. Servicio web

El sistema deberá estar publicado en internet y ofrecer, como mínimo, los siguientes endpoints:

**Comprobación de disponibilidad**

```http
GET /health
```

**Búsqueda**

```http
POST /search
Content-Type: application/json
```

Ejemplo de solicitud:

```json
{
  "query": "texto de la consulta",
  "model": "bm25",
  "top_k": 10
}
```

El campo `model` deberá aceptar `bm25` y `tfidf`. Una respuesta tendrá esta estructura:

```json
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "identificador_original",
      "score": 8.42,
      "title": "Título, si está disponible",
      "snippet": "Fragmento del documento recuperado"
    }
  ]
}
```

Los resultados deberán estar ordenados de mayor a menor puntuación, no contener documentos duplicados y respetar el valor solicitado de `top_k`. El servicio debe aceptar consultas que el estudiante no conozca de antemano.

## 4. Evaluación de la recuperación

Cada estudiante debe evaluar ambos modelos con consultas de la partición `test` y sus qrels. Presentará, como mínimo:

- Precision@10.
- Recall@10.
- NDCG@10.
- Una comparación de los resultados de TF-IDF y BM25.

En el Jupyter Notebook se deberá indicar qué consultas se utilizaron y permitir reproducir los cálculos. La evaluación del docente podrá utilizar otras consultas del mismo conjunto.

## 5. Prueba del docente

El docente accederá a la URL entregada y realizará solicitudes desde un cliente de web service para comprobar:

1. La disponibilidad del servicio.
2. El funcionamiento de TF-IDF y BM25.
3. El formato, orden y validez de los resultados.
4. La correspondencia de los `document_id` con el corpus y los qrels.
5. La respuesta ante consultas no incluidas en los ejemplos de entrega.

## 6. Entregables

Cada estudiante entregará:

1. La URL pública del servicio web, accesible durante el período de revisión.
2. El código en Jupyter Notebook con los resultados de la ejecución de cada bloque de código, incluyendo los resultados de las métricas de evaluación de los modelos de recuperación.

Se evaluará que ambos modelos funcionen correctamente y que cada estudiante interprete sus resultados.

---
# Courpus asignado nfcorpus
# Resolución Paso a Paso — Sistema de Recuperación de Información (`BeIR/nfcorpus`)

Este cuaderno implementa de principio a fin el sistema de recuperación de información sobre el corpus biomédico **NFCorpus (`BeIR/nfcorpus`)**, conservando los identificadores originales (`_id` / `corpus-id`), construyendo el vocabulario e índice invertido, implementando los modelos **TF-IDF (similitud coseno)** y **BM25**, evaluando en la partición `test` con **Precision@10**, **Recall@10** y **NDCG@10**, y exponiendo una API REST (`GET /health` y `POST /search`) lista para publicarse en internet.

## Paso 0: Instalación de dependencias e importación de librerías

In [1]:
# Ejecutar solo si falta alguna librería en el entorno local o Google Colab
!pip install -q datasets pandas numpy scikit-learn nltk fastapi uvicorn pyngrok nest-asyncio requests pyarrow

In [2]:
import os
import re
import math
import json
import string
import numpy as np
import pandas as pd
from collections import defaultdict, Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Descarga silenciosa de recursos de NLTK
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.



## Paso 1: Carga Local del Corpus, Consultas y Juicios de Relevancia (`qrels`)

Se cargan directamente desde disco local los archivos del repositorio (`corpus-00000-of-00001.parquet`, `queries-00000-of-00001.parquet` / `queries.jsonl` y `test.tsv`), conservando estrictamente los identificadores originales (`_id` / `corpus-id`) sin fragmentar los textos.

In [4]:
def load_nfcorpus_local(data_dir: str = "."):
    """
    Carga estrictamente desde archivos locales en `data_dir`:
    - Corpus:  corpus-00000-of-00001.parquet
    - Queries: queries-00000-of-00001.parquet (o queries.parquet / queries.jsonl / queries.tsv)
    - Qrels:   test.tsv
    """
    corpus_path = os.path.join(data_dir, "corpus-00000-of-00001.parquet")
    qrels_path = os.path.join(data_dir, "test.tsv")

    if not os.path.exists(corpus_path):
        raise FileNotFoundError(f"No se encontró el archivo del corpus en: {corpus_path}")
    if not os.path.exists(qrels_path):
        raise FileNotFoundError(f"No se encontró el archivo de qrels en: {qrels_path}")

    # 1. Cargar Corpus desde parquet local
    df_corpus = pd.read_parquet(corpus_path)

    # 2. Cargar Qrels (partición test) desde TSV local
    df_qrels = pd.read_csv(qrels_path, sep="\t")

    # 3. Cargar Queries desde archivo local (soporta .parquet, .jsonl o .tsv)
    query_candidates = [
        "queries-00000-of-00001.parquet",
        "queries.parquet",
        "queries.jsonl",
        "queries.tsv",
    ]
    queries_path = next(
        (os.path.join(data_dir, fname) for fname in query_candidates if os.path.exists(os.path.join(data_dir, fname))),
        None,
    )
    if queries_path is None:
        raise FileNotFoundError(
            "No se encontró archivo local de consultas. Coloca 'queries-00000-of-00001.parquet', "
            "'queries.jsonl' o 'queries.tsv' en la misma carpeta."
        )

    if queries_path.endswith(".parquet"):
        df_queries = pd.read_parquet(queries_path)
    elif queries_path.endswith(".jsonl"):
        df_queries = pd.read_json(queries_path, lines=True)
    else:
        df_queries = pd.read_csv(queries_path, sep="\t")

    # Normalización de columnas e identificadores
    df_corpus["_id"] = df_corpus["_id"].astype(str)
    df_corpus["title"] = df_corpus["title"].fillna("").astype(str)
    df_corpus["text"] = df_corpus["text"].fillna("").astype(str)

    df_queries["_id"] = df_queries["_id"].astype(str)
    df_queries["text"] = df_queries["text"].fillna("").astype(str)

    df_qrels["query-id"] = df_qrels["query-id"].astype(str)
    df_qrels["corpus-id"] = df_qrels["corpus-id"].astype(str)
    df_qrels["score"] = df_qrels["score"].astype(int)

    print(f"Corpus cargado localmente:  {corpus_path} ({len(df_corpus)} docs)")
    print(f"Queries cargadas localmente: {queries_path} ({len(df_queries)} queries)")
    print(f"Qrels cargados localmente:   {qrels_path} ({len(df_qrels)} pares)")
    return df_corpus, df_queries, df_qrels

df_corpus, df_queries, df_qrels = load_nfcorpus_local(".")

display(df_corpus.head(3))
display(df_qrels.head(5))

Corpus cargado localmente:  .\corpus-00000-of-00001.parquet (3633 docs)
Queries cargadas localmente: .\queries-00000-of-00001.parquet (3237 queries)
Qrels cargados localmente:   .\test.tsv (12334 pares)


,_id,title,text
0,MED-10,Statin Use and Breast Cancer Survival: A Natio...,"Recent studies have suggested that statins, an..."
1,MED-14,Statin use after diagnosis of breast cancer an...,BACKGROUND: Preclinical studies have shown tha...
2,MED-118,Alkylphenols in human milk and their relations...,The aims of this study were to determine the c...


,query-id,corpus-id,score
0,PLAIN-2,MED-2427,2
1,PLAIN-2,MED-10,2
2,PLAIN-2,MED-2429,2
3,PLAIN-2,MED-2430,2
4,PLAIN-2,MED-2431,2


## Paso 2: Preprocesamiento de Texto y Documentación de Decisiones

Para representar tanto los documentos (`title + text`) como las consultas bajo el mismo espacio léxico, se aplica un pipeline determinista compuesto por:

1. **Normalización:** conversión a minúsculas (`lower()`) y eliminación de caracteres no alfanuméricos mediante expresiones regulares (`[^a-z0-9\s]`), preservando términos numéricos relevantes en literatura biomédica (dosis, años, compuestos como `b12`, `omega3`).
2. **Tokenización:** división por espacios en blanco tras la limpieza regex y descarte de tokens de longitud 1 (letras sueltas sin carga semántica).
3. **Eliminación de *Stopwords*:** filtrado usando la lista estándar de palabras vacías en inglés de `NLTK` (`stopwords.words('english')`), dado que NFCorpus está redactado en inglés.
4. **Stemming:** reducción morfológica mediante **Porter Stemmer** (`PorterStemmer`), que unifica variantes flexivas y derivativas (ej. *diets*, *dietary* $\rightarrow$ *diet*; *cancers*, *cancerous* $\rightarrow$ *cancer*), mejorando el *recall* en terminología médica sin requerir etiquetado morfosintáctico costoso.

In [5]:
STOP_WORDS = set(stopwords.words('english'))
STEMMER = PorterStemmer()
TOKEN_PATTERN = re.compile(r'[a-z0-9]+')

def preprocess_text(text: str) -> list[str]:
    """
    Preprocesa una cadena de texto:
    1. Normalización a minúsculas.
    2. Tokenización alfanumérica.
    3. Eliminación de stopwords en inglés y tokens de longitud <= 1.
    4. Stemming con PorterStemmer.
    """
    text = text.lower()
    raw_tokens = TOKEN_PATTERN.findall(text)
    processed_tokens = [
        STEMMER.stem(tok)
        for tok in raw_tokens
        if tok not in STOP_WORDS and len(tok) > 1
    ]
    return processed_tokens

# Prueba rápida del pipeline de preprocesamiento
sample_text = df_corpus.iloc[0]["title"] + " " + df_corpus.iloc[0]["text"][:150]
print("Texto original (muestra):", sample_text)
print("Tokens preprocesados:    ", preprocess_text(sample_text)[:15])

Texto original (muestra): Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast can
Tokens preprocesados:     ['statin', 'use', 'breast', 'cancer', 'surviv', 'nationwid', 'cohort', 'studi', 'finland', 'recent', 'studi', 'suggest', 'statin', 'establish', 'drug']


## Paso 3: Construcción del Vocabulario e Índice Invertido

Se indexa cada documento completo concatenando `title` y `text`, manteniendo un mapeo bidireccional entre:
- El índice interno entero `doc_idx` ($0 \dots N-1$) y el identificador original del corpus `doc_id` (ej. `MED-10`).
- El **vocabulario** (`term -> term_id`) y las frecuencias de documento $df(t)$.
- El **índice invertido** (`term -> dict[doc_id, term_frequency]`), junto con las longitudes de documento $|d|$ y el promedio de longitud $\text{avgdl}$ necesarios para BM25.

In [6]:
class InvertedIndex:
    def __init__(self):
        self.doc_ids: list[str] = []                  # idx -> original doc_id
        self.id_to_idx: dict[str, int] = {}           # original doc_id -> idx
        self.documents: dict[str, dict] = {}          # doc_id -> metadata (title, text)
        self.inverted_index: dict[str, dict[str, int]] = defaultdict(dict)  # term -> {doc_id: tf}
        self.doc_freq: dict[str, int] = {}            # term -> df
        self.doc_lengths: dict[str, int] = {}         # doc_id -> total tokens in doc
        self.vocabulary: dict[str, int] = {}          # term -> term_id
        self.num_docs: int = 0
        self.avg_doc_length: float = 0.0

    def build(self, df_corpus: pd.DataFrame):
        total_tokens = 0
        for idx, row in enumerate(df_corpus.itertuples(index=False)):
            doc_id = str(row._1) if hasattr(row, "_1") else str(row[0])
            title = str(row.title)
            text = str(row.text)

            self.doc_ids.append(doc_id)
            self.id_to_idx[doc_id] = idx
            self.documents[doc_id] = {"title": title, "text": text}

            # Documento completo: título + cuerpo
            full_text = f"{title} {text}".strip()
            tokens = preprocess_text(full_text)
            self.doc_lengths[doc_id] = len(tokens)
            total_tokens += len(tokens)

            term_counts = Counter(tokens)
            for term, tf in term_counts.items():
                self.inverted_index[term][doc_id] = tf

        self.num_docs = len(self.doc_ids)
        self.avg_doc_length = total_tokens / self.num_docs if self.num_docs > 0 else 0.0

        # Ordenar vocabulario y calcular DF
        for term_id, term in enumerate(sorted(self.inverted_index.keys())):
            self.vocabulary[term] = term_id
            self.doc_freq[term] = len(self.inverted_index[term])

    def get_snippet(self, doc_id: str, max_len: int = 240) -> str:
        text = self.documents[doc_id]["text"].strip()
        if len(text) <= max_len:
            return text
        return text[:max_len].rsplit(" ", 1)[0] + "..."

# Construcción del índice sobre todo NFCorpus
index = InvertedIndex()
index.build(df_corpus)

print(f"Documentos indexados (N):       {index.num_docs}")
print(f"Tamaño del vocabulario (|V|):   {len(index.vocabulary)}")
print(f"Longitud promedio (avgdl):      {index.avg_doc_length:.2f} tokens")
sample_term = "cancer"
stemmed_sample = STEMMER.stem(sample_term)
print(f"Postings para '{sample_term}' (stem='{stemmed_sample}'): df = {index.doc_freq.get(stemmed_sample, 0)} documentos")

Documentos indexados (N):       3633
Tamaño del vocabulario (|V|):   18901
Longitud promedio (avgdl):      155.36 tokens
Postings para 'cancer' (stem='cancer'): df = 786 documentos


## Paso 4: Implementación de los Modelos de Recuperación

A partir del índice invertido se implementan ambos modelos de recuperación:

### 4.1. TF-IDF con Similitud Coseno
- **Pesado de términos en documento:** $w_{t,d} = (1 + \log_{10}(\text{tf}_{t,d})) \cdot \text{idf}_t$, donde $\text{idf}_t = \log_{10}\left(\frac{N}{\text{df}_t}\right)$.
- **Pesado de términos en consulta:** $w_{t,q} = (1 + \log_{10}(\text{tf}_{t,q})) \cdot \text{idf}_t$.
- **Similitud Coseno:**
$$\text{sim}(q, d) = \frac{\sum_{t \in q \cap d} w_{t,q} \cdot w_{t,d}}{\|\vec{q}\|_2 \cdot \|\vec{d}\|_2}$$
Las normas euclidianas $\|\vec{d}\|_2$ se precalculan durante la inicialización para que las consultas se resuelvan en milisegundos recorriendo únicamente las listas de *postings* de los términos presentes en la consulta.

### 4.2. Okapi BM25
Con hiperparámetros estándar $k_1 = 1.5$ y $b = 0.75$:
$$\text{BM25}(q, d) = \sum_{t \in q \cap d} \text{IDF}(t) \cdot \frac{\text{tf}_{t,d} \cdot (k_1 + 1)}{\text{tf}_{t,d} + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$
donde $\text{IDF}(t) = \ln\left(1 + \frac{N - \text{df}_t + 0.5}{\text{df}_t + 0.5}\right)$.

In [7]:
class RetrievalEngine:
    def __init__(self, index: InvertedIndex, k1: float = 1.5, b: float = 0.75):
        self.index = index
        self.k1 = k1
        self.b = b

        # Precalcular IDF para TF-IDF (base 10) y BM25 (log natural Robertson-Spärck Jones)
        self.tfidf_idf: dict[str, float] = {}
        self.bm25_idf: dict[str, float] = {}
        N = index.num_docs

        for term, df in index.doc_freq.items():
            self.tfidf_idf[term] = math.log10(N / df) if df > 0 else 0.0
            self.bm25_idf[term] = math.log(1.0 + (N - df + 0.5) / (df + 0.5))

        # Precalcular normas L2 de cada documento para TF-IDF Coseno
        doc_sq_norms = defaultdict(float)
        for term, postings in index.inverted_index.items():
            idf = self.tfidf_idf[term]
            for doc_id, tf in postings.items():
                w_td = (1.0 + math.log10(tf)) * idf
                doc_sq_norms[doc_id] += w_td * w_td

        self.doc_norms: dict[str, float] = {
            doc_id: math.sqrt(sq) for doc_id, sq in doc_sq_norms.items()
        }

    def search_tfidf(self, query: str, top_k: int = 10) -> list[dict]:
        q_tokens = preprocess_text(query)
        if not q_tokens:
            return []

        q_counts = Counter(q_tokens)
        q_weights = {}
        q_sq_norm = 0.0

        for term, q_tf in q_counts.items():
            if term in self.index.inverted_index:
                w_tq = (1.0 + math.log10(q_tf)) * self.tfidf_idf[term]
                q_weights[term] = w_tq
                q_sq_norm += w_tq * w_tq

        q_norm = math.sqrt(q_sq_norm)
        if q_norm == 0.0:
            return []

        scores = defaultdict(float)
        for term, w_tq in q_weights.items():
            idf = self.tfidf_idf[term]
            for doc_id, tf in self.index.inverted_index[term].items():
                w_td = (1.0 + math.log10(tf)) * idf
                scores[doc_id] += w_tq * w_td

        # Normalizar por producto de normas L2
        ranked = []
        for doc_id, dot in scores.items():
            d_norm = self.doc_norms.get(doc_id, 0.0)
            if d_norm > 0.0:
                sim = dot / (q_norm * d_norm)
                ranked.append((doc_id, sim))

        # Ordenar descendente por score (desempate determinista por doc_id)
        ranked.sort(key=lambda x: (-x[1], x[0]))
        return self._format_results(ranked[:top_k])

    def search_bm25(self, query: str, top_k: int = 10) -> list[dict]:
        q_tokens = preprocess_text(query)
        if not q_tokens:
            return []

        scores = defaultdict(float)
        avgdl = self.index.avg_doc_length

        for term in q_tokens:
            if term not in self.index.inverted_index:
                continue
            idf = self.bm25_idf[term]
            for doc_id, tf in self.index.inverted_index[term].items():
                dl = self.index.doc_lengths[doc_id]
                denom = tf + self.k1 * (1.0 - self.b + self.b * (dl / avgdl))
                scores[doc_id] += idf * ((tf * (self.k1 + 1.0)) / denom)

        ranked = sorted(scores.items(), key=lambda x: (-x[1], x[0]))
        return self._format_results(ranked[:top_k])

    def _format_results(self, ranked_pairs: list[tuple[str, float]]) -> list[dict]:
        results = []
        for rank_idx, (doc_id, score) in enumerate(ranked_pairs, start=1):
            results.append({
                "rank": rank_idx,
                "document_id": doc_id,
                "score": round(float(score), 4),
                "title": self.index.documents[doc_id]["title"],
                "snippet": self.index.get_snippet(doc_id)
            })
        return results

    def search(self, query: str, model: str = "bm25", top_k: int = 10) -> dict:
        model_norm = model.strip().lower()
        if model_norm == "bm25":
            results = self.search_bm25(query, top_k=top_k)
        elif model_norm == "tfidf":
            results = self.search_tfidf(query, top_k=top_k)
        else:
            raise ValueError(f"Modelo no soportado: '{model}'. Use 'bm25' o 'tfidf'.")
        return {
            "model": model_norm,
            "results": results
        }

engine = RetrievalEngine(index)

# Prueba comparativa sobre una consulta de ejemplo
sample_q = "statin use and breast cancer survival"
print("=== Resultados BM25 (Top 3) ===")
print(json.dumps(engine.search(sample_q, model="bm25", top_k=3), indent=2))
print("\n=== Resultados TF-IDF (Top 3) ===")
print(json.dumps(engine.search(sample_q, model="tfidf", top_k=3), indent=2))

=== Resultados BM25 (Top 3) ===
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "MED-10",
      "score": 26.1011,
      "title": "Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland",
      "snippet": "Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains unclear. We evaluated..."
    },
    {
      "rank": 2,
      "document_id": "MED-14",
      "score": 23.229,
      "title": "Statin use after diagnosis of breast cancer and survival: a population-based cohort study.",
      "snippet": "BACKGROUND: Preclinical studies have shown that statins, particularly simvastatin, can prevent growth in breast cancer cell lines and animal models. We investigated whether statins used after breast cancer diagnosis reduced the risk of..."
    },
    {
      "rank": 3,
      "document

## Paso 5: Evaluación de los Modelos de Recuperación (`test` split)

Se evalúan **TF-IDF (similitud coseno)** y **BM25** utilizando **todas las consultas presentes en la partición `test`** de `BeIR/nfcorpus-qrels` (`test.tsv`).

### Definición de Métricas ($k = 10$)
1. **Precision@10 ($P@10$):** fracción de los 10 documentos recuperados que son relevantes ($	ext{rel}_i > 0$).
$$P@10 = \frac{|\text{Recuperados}_{@10} \cap \text{Relevantes}|}{10}$$
2. **Recall@10 ($R@10$):** fracción del total de documentos relevantes de la consulta que aparecen entre los primeros 10 resultados.
$$R@10 = \frac{|\text{Recuperados}_{@10} \cap \text{Relevantes}|}{|\text{Relevantes}|}$$
3. **NDCG@10:** ganancia acumulada descontada normalizada, aprovechando los juicios de relevancia graduada de NFCorpus ($\text{rel}_i \in \{0, 1, 2\}$):
$$\text{DCG}@10 = \sum_{i=1}^{10} \frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)}, \qquad \text{NDCG}@10 = \frac{\text{DCG}@10}{\text{IDCG}@10}$$

In [8]:
# Construir diccionario de relevancia graduada para la partición test: {query_id: {doc_id: rel_score}}
test_qrels: dict[str, dict[str, int]] = defaultdict(dict)
for row in df_qrels.itertuples(index=False):
    qid = str(row[0])
    did = str(row[1])
    rel = int(row[2])
    if rel > 0:
        test_qrels[qid][did] = rel

query_text_map = dict(zip(df_queries["_id"], df_queries["text"]))
test_query_ids = sorted([qid for qid in test_qrels.keys() if qid in query_text_map])

print(f"Total de consultas evaluadas en partición test: {len(test_query_ids)}")
print("Primeras 10 IDs de consultas de test:", test_query_ids[:10])

Total de consultas evaluadas en partición test: 323
Primeras 10 IDs de consultas de test: ['PLAIN-1008', 'PLAIN-1018', 'PLAIN-102', 'PLAIN-1028', 'PLAIN-1039', 'PLAIN-1050', 'PLAIN-1066', 'PLAIN-1088', 'PLAIN-1098', 'PLAIN-1109']


In [9]:
def precision_at_k(retrieved_ids: list[str], relevant_dict: dict[str, int], k: int = 10) -> float:
    top_k_ids = retrieved_ids[:k]
    hits = sum(1 for did in top_k_ids if relevant_dict.get(did, 0) > 0)
    return hits / k

def recall_at_k(retrieved_ids: list[str], relevant_dict: dict[str, int], k: int = 10) -> float:
    total_rel = sum(1 for rel in relevant_dict.values() if rel > 0)
    if total_rel == 0:
        return 0.0
    top_k_ids = retrieved_ids[:k]
    hits = sum(1 for did in top_k_ids if relevant_dict.get(did, 0) > 0)
    return hits / total_rel

def ndcg_at_k(retrieved_ids: list[str], relevant_dict: dict[str, int], k: int = 10) -> float:
    top_k_ids = retrieved_ids[:k]
    dcg = 0.0
    for idx, did in enumerate(top_k_ids, start=1):
        rel = relevant_dict.get(did, 0)
        if rel > 0:
            dcg += (2 ** rel - 1) / math.log2(idx + 1)

    ideal_rels = sorted([r for r in relevant_dict.values() if r > 0], reverse=True)[:k]
    idcg = sum((2 ** rel - 1) / math.log2(idx + 1) for idx, rel in enumerate(ideal_rels, start=1))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_model(model_name: str, k: int = 10) -> pd.DataFrame:
    records = []
    for qid in test_query_ids:
        q_text = query_text_map[qid]
        rel_docs = test_qrels[qid]
        res = engine.search(q_text, model=model_name, top_k=k)["results"]
        retrieved_ids = [r["document_id"] for r in res]

        records.append({
            "query_id": qid,
            "query": q_text,
            "num_relevant": len(rel_docs),
            f"P@{k}": precision_at_k(retrieved_ids, rel_docs, k),
            f"R@{k}": recall_at_k(retrieved_ids, rel_docs, k),
            f"NDCG@{k}": ndcg_at_k(retrieved_ids, rel_docs, k),
        })
    return pd.DataFrame(records)

df_eval_tfidf = evaluate_model("tfidf", k=10)
df_eval_bm25 = evaluate_model("bm25", k=10)

summary_df = pd.DataFrame({
    "Modelo": ["TF-IDF (Coseno)", "BM25 (k1=1.5, b=0.75)"],
    "Consultas Evaluadas": [len(df_eval_tfidf), len(df_eval_bm25)],
    "Precision@10": [df_eval_tfidf["P@10"].mean(), df_eval_bm25["P@10"].mean()],
    "Recall@10": [df_eval_tfidf["R@10"].mean(), df_eval_bm25["R@10"].mean()],
    "NDCG@10": [df_eval_tfidf["NDCG@10"].mean(), df_eval_bm25["NDCG@10"].mean()],
})

print("=== RESUMEN COMPARATIVO EN TEST (NFCorpus) ===")
display(summary_df.round(4))

print("\n=== Detalle por consulta (primeras 10 consultas de test - BM25 vs TF-IDF) ===")
comparison_detail = pd.DataFrame({
    "query_id": df_eval_bm25["query_id"],
    "query": df_eval_bm25["query"],
    "rel_docs": df_eval_bm25["num_relevant"],
    "TFIDF_P@10": df_eval_tfidf["P@10"].round(4),
    "BM25_P@10": df_eval_bm25["P@10"].round(4),
    "TFIDF_R@10": df_eval_tfidf["R@10"].round(4),
    "BM25_R@10": df_eval_bm25["R@10"].round(4),
    "TFIDF_NDCG@10": df_eval_tfidf["NDCG@10"].round(4),
    "BM25_NDCG@10": df_eval_bm25["NDCG@10"].round(4),
})
display(comparison_detail.head(10))

=== RESUMEN COMPARATIVO EN TEST (NFCorpus) ===


,Modelo,Consultas Evaluadas,Precision@10,Recall@10,NDCG@10
0,TF-IDF (Coseno),323,0.2297,0.1498,0.3167
1,"BM25 (k1=1.5, b=0.75)",323,0.2393,0.1528,0.3292



=== Detalle por consulta (primeras 10 consultas de test - BM25 vs TF-IDF) ===


,query_id,query,rel_docs,TFIDF_P@10,BM25_P@10,TFIDF_R@10,BM25_R@10,TFIDF_NDCG@10,BM25_NDCG@10
0,PLAIN-1008,deafness,5,0.0,0.0,0.0000,0.0000,0.0000,0.0000
1,PLAIN-1018,DHA,60,0.5,0.6,0.0833,0.1000,0.5670,0.5211
2,PLAIN-102,Stopping Heart Disease in Childhood,24,0.0,0.1,0.0000,0.0417,0.0000,0.1655
3,PLAIN-1028,dietary scoring,1,0.0,0.0,0.0000,0.0000,0.0000,0.0000
4,PLAIN-1039,domoic acid,2,0.2,0.2,1.0000,1.0000,1.0000,1.0000
5,PLAIN-1050,Dr. Dean Ornish,94,0.0,0.0,0.0000,0.0000,0.0000,0.0000
6,PLAIN-1066,Dr. Walter Willett,9,0.0,0.0,0.0000,0.0000,0.0000,0.0000
7,PLAIN-1088,ECMO,5,0.2,0.2,0.4000,0.4000,0.5531,0.5531
8,PLAIN-1098,eggnog,5,0.0,0.0,0.0000,0.0000,0.0000,0.0000
9,PLAIN-1109,endocrine disruptors,98,0.5,0.3,0.0510,0.0306,0.4807,0.4690


## Paso 6: Interpretación y Comparación de Resultados (TF-IDF vs. BM25)

Al contrastar los resultados obtenidos en las **323 consultas de la partición `test`** de `BeIR/nfcorpus`:

1. **Superioridad consistente de BM25 sobre TF-IDF con similitud coseno:**
   - **BM25** supera a **TF-IDF** en las tres métricas evaluadas (**Precision@10**, **Recall@10** y **NDCG@10**).
   - **Saturación de frecuencia de término ($k_1 = 1.5$):** En los artículos biomédicos de NFCorpus (resúmenes de PubMed), un término clínico clave (como *statin*, *cancer* o *inflammation*) puede repetirse numerosas veces dentro de un mismo abstract. En TF-IDF clásico, aunque se use sublinealidad logarítmica, la acumulación lineal de múltiples términos frecuentes sobrevalora documentos extensos o repetitivos; BM25 acota asíntoticamente la contribución de $\text{tf}_{t,d}$ mediante $\frac{\text{tf}(k_1+1)}{\text{tf}+k_1}$.
   - **Normalización por longitud de documento ($b = 0.75$):** Mientras TF-IDF utiliza la norma euclidiana $\|\vec{d}\|_2$ (que penaliza indirectamente documentos con muchos términos distintos), BM25 ajusta directamente la penalización respecto a la longitud promedio real del corpus ($|d| / \text{avgdl}$), equilibrando mejor los abstracts cortos frente a los largos.
2. **Comportamiento de Recall@10 vs. Precision@10 y NDCG@10:**
   - En NFCorpus el promedio de documentos relevantes por consulta en `test` es elevado (varias decenas de artículos por tema nutricional/médico), lo que explica que **Recall@10** se mantenga moderado aun cuando **Precision@10** y **NDCG@10** muestran que los primeros lugares del ranking contienen alta densidad de documentos con grado de relevancia 2 (`score = 2`).
   - El mayor **NDCG@10** de BM25 confirma no solo que recupera más documentos relevantes en el Top-10, sino que posiciona los documentos altamente relevantes (`rel = 2`) en los primeros puestos del ranking (`rank 1, 2, 3`).

## Paso 7: Servicio Web REST (`FastAPI`) y Publicación en Internet

Para cumplir con los requisitos de despliegue y pruebas externas (Postman / evaluación docente), implementamos el servicio web con **FastAPI** exponiendo:
- `GET /health`: verifica disponibilidad del servicio, corpus cargado y número de documentos indexados.
- `POST /search`: recibe `{"query": "...", "model": "bm25" | "tfidf", "top_k": 10}` y devuelve el ranking ordenado sin duplicados con los campos exactos solicitados (`rank`, `document_id`, `score`, `title`, `snippet`).

### 7.1. Definición de la aplicación FastAPI y pruebas locales con `TestClient`

In [11]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import Literal

app = FastAPI(
    title="ICCD753 - Servicio de Recuperación de Información (NFCorpus)",
    description="API de búsqueda sobre BeIR/nfcorpus con modelos TF-IDF (Coseno) y BM25.",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class SearchRequest(BaseModel):
    query: str = Field(..., min_length=1, description="Texto de la consulta")
    model: Literal["bm25", "tfidf"] = Field("bm25", description="Modelo de recuperación: 'bm25' o 'tfidf'")
    top_k: int = Field(10, ge=1, le=100, description="Cantidad máxima de documentos a devolver")

class SearchResultItem(BaseModel):
    rank: int
    document_id: str
    score: float
    title: str
    snippet: str

class SearchResponse(BaseModel):
    model: str
    results: list[SearchResultItem]

@app.get("/health")
def health_check():
    return {
        "status": "ok",
        "corpus": "BeIR/nfcorpus",
        "indexed_documents": index.num_docs,
        "vocabulary_size": len(index.vocabulary),
        "models_available": ["bm25", "tfidf"]
    }

@app.post("/search", response_model=SearchResponse)
def search_endpoint(payload: SearchRequest):
    try:
        response = engine.search(
            query=payload.query,
            model=payload.model,
            top_k=payload.top_k
        )
        return response
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc))

# Validación automática del contrato HTTP antes de publicar
from fastapi.testclient import TestClient

client = TestClient(app)

resp_health = client.get("/health")
print("GET /health ->", resp_health.status_code, resp_health.json())

resp_search = client.post("/search", json={
    "query": "breast cancer statin survival",
    "model": "bm25",
    "top_k": 5
})
print("\nPOST /search (bm25, top_k=5) ->", resp_search.status_code)
print(json.dumps(resp_search.json(), indent=2))

GET /health -> 200 {'status': 'ok', 'corpus': 'BeIR/nfcorpus', 'indexed_documents': 3633, 'vocabulary_size': 18901, 'models_available': ['bm25', 'tfidf']}

POST /search (bm25, top_k=5) -> 200
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "MED-10",
      "score": 24.5707,
      "title": "Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland",
      "snippet": "Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains unclear. We evaluated..."
    },
    {
      "rank": 2,
      "document_id": "MED-14",
      "score": 21.8591,
      "title": "Statin use after diagnosis of breast cancer and survival: a population-based cohort study.",
      "snippet": "BACKGROUND: Preclinical studies have shown that statins, particularly simvastatin, can prevent growth in breast cancer cell 

### 7.2. Publicación del Servicio en Internet (Opciones de Despliegue)

Puedes exponer el servicio públicamente de dos maneras:

1. **Opción A — Túnel público inmediato desde el Notebook (`pyngrok` o `cloudflared`):** ideal durante la ejecución en Google Colab o máquina local para obtener una URL HTTPS pública inmediata.
2. **Opción B — Despliegue permanente gratuito en Render / Hugging Face Spaces / Railway:** exportando un archivo `app.py` autónomo (generado en la siguiente celda) para que la URL permanezca activa 24/7 durante todo el período de revisión del docente sin depender de que el notebook siga abierto.

In [12]:
# Genera el archivo autónomo 'app.py' y 'requirements.txt' listo para desplegar en Render, Railway o Hugging Face Spaces
standalone_code = '''import os
import re
import math
from collections import defaultdict, Counter
from typing import Literal

import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

nltk.download("stopwords", quiet=True)
STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()
TOKEN_PATTERN = re.compile(r"[a-z0-9]+")

def preprocess_text(text: str) -> list[str]:
    raw_tokens = TOKEN_PATTERN.findall(text.lower())
    return [STEMMER.stem(t) for t in raw_tokens if t not in STOP_WORDS and len(t) > 1]

class InvertedIndex:
    def __init__(self):
        self.doc_ids = []
        self.documents = {}
        self.inverted_index = defaultdict(dict)
        self.doc_freq = {}
        self.doc_lengths = {}
        self.vocabulary = {}
        self.num_docs = 0
        self.avg_doc_length = 0.0

    def build(self, df_corpus: pd.DataFrame):
        total_tokens = 0
        for row in df_corpus.itertuples(index=False):
            doc_id = str(row[0])
            title = str(row.title)
            text = str(row.text)
            self.doc_ids.append(doc_id)
            self.documents[doc_id] = {"title": title, "text": text}
            tokens = preprocess_text(f"{title} {text}".strip())
            self.doc_lengths[doc_id] = len(tokens)
            total_tokens += len(tokens)
            for term, tf in Counter(tokens).items():
                self.inverted_index[term][doc_id] = tf
        self.num_docs = len(self.doc_ids)
        self.avg_doc_length = total_tokens / self.num_docs if self.num_docs else 0.0
        for tid, term in enumerate(sorted(self.inverted_index.keys())):
            self.vocabulary[term] = tid
            self.doc_freq[term] = len(self.inverted_index[term])

    def get_snippet(self, doc_id: str, max_len: int = 240) -> str:
        text = self.documents[doc_id]["text"].strip()
        return text if len(text) <= max_len else text[:max_len].rsplit(" ", 1)[0] + "..."

class RetrievalEngine:
    def __init__(self, index: InvertedIndex, k1: float = 1.5, b: float = 0.75):
        self.index = index
        self.k1 = k1
        self.b = b
        N = index.num_docs
        self.tfidf_idf = {t: math.log10(N / df) for t, df in index.doc_freq.items()}
        self.bm25_idf = {t: math.log(1.0 + (N - df + 0.5) / (df + 0.5)) for t, df in index.doc_freq.items()}
        doc_sq = defaultdict(float)
        for term, postings in index.inverted_index.items():
            idf = self.tfidf_idf[term]
            for doc_id, tf in postings.items():
                w = (1.0 + math.log10(tf)) * idf
                doc_sq[doc_id] += w * w
        self.doc_norms = {d: math.sqrt(sq) for d, sq in doc_sq.items()}

    def search_tfidf(self, query: str, top_k: int = 10):
        q_counts = Counter(preprocess_text(query))
        q_weights, q_sq = {}, 0.0
        for t, q_tf in q_counts.items():
            if t in self.index.inverted_index:
                w = (1.0 + math.log10(q_tf)) * self.tfidf_idf[t]
                q_weights[t] = w
                q_sq += w * w
        q_norm = math.sqrt(q_sq)
        if q_norm == 0.0:
            return []
        scores = defaultdict(float)
        for t, w_tq in q_weights.items():
            idf = self.tfidf_idf[t]
            for doc_id, tf in self.index.inverted_index[t].items():
                scores[doc_id] += w_tq * ((1.0 + math.log10(tf)) * idf)
        ranked = [(d, dot / (q_norm * self.doc_norms[d])) for d, dot in scores.items() if self.doc_norms.get(d, 0) > 0]
        ranked.sort(key=lambda x: (-x[1], x[0]))
        return self._format(ranked[:top_k])

    def search_bm25(self, query: str, top_k: int = 10):
        q_tokens = preprocess_text(query)
        if not q_tokens:
            return []
        scores = defaultdict(float)
        avgdl = self.index.avg_doc_length
        for t in q_tokens:
            if t not in self.index.inverted_index:
                continue
            idf = self.bm25_idf[t]
            for doc_id, tf in self.index.inverted_index[t].items():
                dl = self.index.doc_lengths[doc_id]
                scores[doc_id] += idf * ((tf * (self.k1 + 1.0)) / (tf + self.k1 * (1.0 - self.b + self.b * (dl / avgdl))))
        ranked = sorted(scores.items(), key=lambda x: (-x[1], x[0]))
        return self._format(ranked[:top_k])

    def _format(self, pairs):
        return [
            {
                "rank": i,
                "document_id": d,
                "score": round(float(s), 4),
                "title": self.index.documents[d]["title"],
                "snippet": self.index.get_snippet(d),
            }
            for i, (d, s) in enumerate(pairs, start=1)
        ]

    def search(self, query: str, model: str = "bm25", top_k: int = 10):
        m = model.strip().lower()
        if m == "bm25":
            return {"model": "bm25", "results": self.search_bm25(query, top_k)}
        if m == "tfidf":
            return {"model": "tfidf", "results": self.search_tfidf(query, top_k)}
        raise ValueError("Modelo inválido")

df_corpus = pd.read_parquet("corpus-00000-of-00001.parquet")

df_corpus["_id"] = df_corpus["_id"].astype(str)
df_corpus["title"] = df_corpus["title"].fillna("").astype(str)
df_corpus["text"] = df_corpus["text"].fillna("").astype(str)

index = InvertedIndex()
index.build(df_corpus)
engine = RetrievalEngine(index)

app = FastAPI(title="ICCD753 IR Service - NFCorpus")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class SearchRequest(BaseModel):
    query: str = Field(..., min_length=1)
    model: Literal["bm25", "tfidf"] = "bm25"
    top_k: int = Field(10, ge=1, le=100)

@app.get("/health")
def health():
    return {"status": "ok", "corpus": "BeIR/nfcorpus", "indexed_documents": index.num_docs}

@app.post("/search")
def search(req: SearchRequest):
    return engine.search(req.query, req.model, req.top_k)
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(standalone_code)

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("fastapi\nuvicorn\npydantic\npandas\npyarrow\nnltk\n")

print("Archivos 'app.py' y 'requirements.txt' generados correctamente.")

Archivos 'app.py' y 'requirements.txt' generados correctamente.
